# Ejercicio 5 – Incorporación de los datos de 2024

En este notebook se incorporan los viajes de taxis amarillos y verdes de **2024** al sistema construido con los datos de 2026. Se verifica que:

1. el sistema de descarga obtenga 2024 desde la fuente original sin volver a descargar 2026;
2. los nuevos archivos estén completos y sean legibles;
3. DuckDB pueda consultar 2024 y 2026 de forma conjunta;
4. las consultas de los Ejercicios 3 y 4 sigan funcionando con el conjunto ampliado.

Igual que en el Ejercicio 4, la conexión de DuckDB se limita a 1 GB de memoria y 2 hilos.

In [1]:
import os
import time
from pathlib import Path

import duckdb
import pandas as pd

# Se trabaja desde la raiz del proyecto para que las rutas data/raw/... funcionen
if Path.cwd().name == "notebooks":
    os.chdir("..")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

DIR_SQL = Path("sql/05_incorporacion")

# Limites de recursos para equipos con poca RAM
con = duckdb.connect(config={"memory_limit": "1GB", "threads": 2, "preserve_insertion_order": False})


def consulta(ruta: Path) -> pd.DataFrame | None:
    """Muestra y ejecuta un archivo SQL; devuelve el resultado como DataFrame (None si no hay resultado)."""
    sql = Path(ruta).read_text()
    print(sql)
    resultado = con.sql(sql)
    return resultado.df() if resultado is not None else None

## 5.1–5.4 Descarga de 2024

El script `scripts/download_data.py` ya acepta varios años con el argumento `--anio`, por lo que **no fue necesario modificar el código**. Basta con indicar los años:

```bash
python scripts/download_data.py --anio 2024 2026
```

La primera ejecución (fuera de este notebook) descargó los 24 archivos de 2024 y omitió los 16 de 2026. A continuación se ejecuta el mismo comando otra vez para comprobar que ningún archivo existente se vuelve a descargar.

In [2]:
!python scripts/download_data.py --anio 2024 2026


=== YELLOW 2024 ===
  2024-01  ya existe, se omite
  2024-02  ya existe, se omite
  2024-03  ya existe, se omite
  2024-04  ya existe, se omite
  2024-05  ya existe, se omite
  2024-06  ya existe, se omite
  2024-07  ya existe, se omite
  2024-08  ya existe, se omite
  2024-09  ya existe, se omite
  2024-10  ya existe, se omite
  2024-11  ya existe, se omite
  2024-12  ya existe, se omite

=== GREEN 2024 ===
  2024-01  ya existe, se omite
  2024-02  ya existe, se omite
  2024-03  ya existe, se omite
  2024-04  ya existe, se omite
  2024-05  ya existe, se omite
  2024-06  ya existe, se omite
  2024-07  ya existe, se omite
  2024-08  ya existe, se omite
  2024-09  ya existe, se omite
  2024-10  ya existe, se omite
  2024-11  ya existe, se omite
  2024-12  ya existe, se omite

=== YELLOW 2026 ===
  2026-01  ya existe, se omite
  2026-02  ya existe, se omite
  2026-03  ya existe, se omite
  2026-04  ya existe, se omite
  2026-05  ya existe, se omite
  2026-06  ya existe, se omite
  2026-0

  2026-09  aun no publicado por la TLC


  2026-10  aun no publicado por la TLC


  2026-11  aun no publicado por la TLC


  2026-12  aun no publicado por la TLC

=== GREEN 2026 ===
  2026-01  ya existe, se omite
  2026-02  ya existe, se omite
  2026-03  ya existe, se omite
  2026-04  ya existe, se omite
  2026-05  ya existe, se omite
  2026-06  ya existe, se omite
  2026-07  ya existe, se omite
  2026-08  ya existe, se omite


  2026-09  aun no publicado por la TLC


  2026-10  aun no publicado por la TLC


  2026-11  aun no publicado por la TLC


  2026-12  aun no publicado por la TLC

RESUMEN
  descargados   : 0
  ya existian   : 40
  no publicados : 8
      yellow 2026-09, yellow 2026-10, yellow 2026-11, yellow 2026-12, green 2026-09, green 2026-10, green 2026-11, green 2026-12
  fallidos      : 0


**Resultado:**

| Ejecución | Descargados | Ya existían | No publicados | Fallidos |
|-----------|------------:|------------:|--------------:|---------:|
| 1ª (descarga de 2024) | 24 | 16 | 8 | 0 |
| 2ª (esta celda) | 0 | 40 | 8 | 0 |

- Los 16 archivos de 2026 se conservaron y no se volvieron a descargar.
- Los 8 "no publicados" corresponden a septiembre–diciembre de 2026, que la TLC aún no publica.
- Además, se comprobó que el tamaño de los 24 archivos de 2024 coincide con el `Content-Length` informado por el servidor.

## 5.5 Verificación de los archivos incorporados

| Consulta | Objetivo |
|----------|----------|
| `sql/05_incorporacion/01_archivos_por_anio.sql` | Contar archivos por tipo y año, con el primer y el último mes. |
| `sql/05_incorporacion/02_registros_por_anio.sql` | Contar registros por tipo y año a partir de los metadatos Parquet. |

Fuente: `data/raw/*/*/*.parquet`.

In [3]:
consulta(DIR_SQL / "01_archivos_por_anio.sql")

-- Ejercicio 5 - Archivos disponibles por tipo de taxi y anio, con el primer y ultimo mes.
-- Fuente: data/raw/*/*/*.parquet.
SELECT
    split_part(file, '/', 3)                                    AS tipo,
    split_part(file, '/', 4)                                    AS anio,
    count(*)                                                    AS archivos,
    min(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1))      AS primer_mes,
    max(regexp_extract(file, '(\d{4}-\d{2})\.parquet', 1))      AS ultimo_mes
FROM glob('data/raw/*/*/*.parquet')
GROUP BY tipo, anio
ORDER BY tipo, anio;



,tipo,anio,archivos,primer_mes,ultimo_mes
0,green,2024,12,2024-01,2024-12
1,green,2026,8,2026-01,2026-08
2,yellow,2024,12,2024-01,2024-12
3,yellow,2026,8,2026-01,2026-08


In [4]:
consulta(DIR_SQL / "02_registros_por_anio.sql")

-- Ejercicio 5 - Registros por tipo de taxi y anio segun los metadatos Parquet (no lee los datos).
-- Fuente: data/raw/*/*/*.parquet.
SELECT
    split_part(file_name, '/', 3)   AS tipo,
    split_part(file_name, '/', 4)   AS anio,
    sum(num_rows)                   AS registros
FROM parquet_file_metadata('data/raw/*/*/*.parquet')
GROUP BY tipo, anio
ORDER BY tipo, anio;



,tipo,anio,registros
0,green,2024,660218.0
1,green,2026,337114.0
2,yellow,2024,41169720.0
3,yellow,2026,29703355.0


**Resultado:**

- 2024 tiene **12 archivos por tipo de taxi** (enero–diciembre) y 2026 conserva sus 8 (enero–agosto): 40 archivos en total.
- 2024 aporta 41,169,720 registros amarillos y 660,218 verdes, que DuckDB lee sin errores.

## Comparación de esquemas entre años

Antes de consultar ambos años juntos, se revisa si los archivos de 2024 tienen las mismas columnas y tipos que los de 2026.

| Consulta | Objetivo |
|----------|----------|
| `sql/05_incorporacion/03_esquema_por_anio.sql` | Contar en cuántos archivos de cada tipo/año aparece cada columna. |
| `sql/05_incorporacion/04_tipos_por_anio.sql` | Listar las columnas cuyo tipo de dato cambia entre años. |
| `sql/05_incorporacion/06_cbd_por_anio.sql` | Contar los valores nulos de `cbd_congestion_fee` por tipo y año. |

Fuente: `data/raw/*/*/*.parquet`.

In [5]:
consulta(DIR_SQL / "03_esquema_por_anio.sql")

-- Ejercicio 5 - Columnas y tipos fisicos Parquet por tipo de taxi y anio.
-- Fuente: data/raw/*/*/*.parquet (esquema).
-- Salida: una fila por columna; cada celda indica en cuantos archivos de ese tipo/anio aparece.
PIVOT (
    SELECT
        name                                                                  AS columna,
        split_part(file_name, '/', 3) || '_' || split_part(file_name, '/', 4) AS tipo_anio,
        file_name
    FROM parquet_schema('data/raw/*/*/*.parquet')
    WHERE num_children IS NULL
)
ON tipo_anio
USING count(file_name)
ORDER BY columna;



,columna,green_2024,green_2026,yellow_2024,yellow_2026
0,Airport_fee,0,0,12,8
1,DOLocationID,12,8,12,8
2,PULocationID,12,8,12,8
3,RatecodeID,12,8,12,8
4,VendorID,12,8,12,8
5,cbd_congestion_fee,0,8,0,8
6,congestion_surcharge,12,8,12,8
7,ehail_fee,12,8,0,0
8,extra,12,8,12,8
9,fare_amount,12,8,12,8


In [6]:
consulta(DIR_SQL / "04_tipos_por_anio.sql")

-- Ejercicio 5 - Columnas cuyo tipo fisico Parquet difiere entre anios para un mismo tipo de taxi.
-- Fuente: data/raw/*/*/*.parquet (esquema).
SELECT
    split_part(file_name, '/', 3)                                       AS tipo,
    name                                                                AS columna,
    string_agg(DISTINCT split_part(file_name, '/', 4) || ': ' || type, ', ' ORDER BY split_part(file_name, '/', 4) || ': ' || type) AS tipos_por_anio
FROM parquet_schema('data/raw/*/*/*.parquet')
WHERE num_children IS NULL
GROUP BY tipo, columna
HAVING count(DISTINCT type) > 1
ORDER BY tipo, columna;



,tipo,columna,tipos_por_anio


In [7]:
consulta(DIR_SQL / "06_cbd_por_anio.sql")

-- Ejercicio 5 - Valores nulos de cbd_congestion_fee por tipo de taxi y anio del archivo.
-- Fuente: data/raw/*/*/*.parquet. Con union_by_name, una columna ausente en un archivo se lee como NULL.
SELECT
    split_part(filename, '/', 3)                                                    AS tipo,
    split_part(filename, '/', 4)                                                    AS anio,
    count(*)                                                                        AS registros,
    count(*) FILTER (WHERE cbd_congestion_fee IS NULL)                              AS cbd_nulos,
    round(100.0 * count(*) FILTER (WHERE cbd_congestion_fee IS NULL) / count(*), 2) AS pct_cbd_nulos
FROM read_parquet('data/raw/*/*/*.parquet', union_by_name = true, filename = true)
GROUP BY tipo, anio
ORDER BY tipo, anio;



,tipo,anio,registros,cbd_nulos,pct_cbd_nulos
0,green,2024,660218,660218,100.0
1,green,2026,337114,0,0.0
2,yellow,2024,41169720,41169720,100.0
3,yellow,2026,29703355,0,0.0


**Resultado:**

- Los tipos de datos **no cambian** entre años (la consulta de tipos no devuelve filas).
- `cbd_congestion_fee` **no existe en 2024**: el 100% de los registros de 2024 la tienen nula al leerla con `union_by_name`. Es el recargo por congestión en el centro de Manhattan, que se empezó a cobrar en 2025.
- `request_source` solo existe desde junio de 2026, como se vio en el Ejercicio 3.

**Decisión:** en la vista `viajes` se reemplazó `cbd_congestion_fee` por `coalesce(cbd_congestion_fee, 0)`, porque en 2024 ese recargo no existía (equivale a 0). Sin este cambio, las consultas que suman los componentes del cobro (`09_atipicos.sql` y `10_total_no_cuadra.sql`) obtendrían `NULL` en todos los viajes de 2024 y los excluirían de los conteos sin avisar. Es el **único cambio** necesario y se hizo en un solo lugar (`sql/04_eda/00_vista_viajes.sql`).

## 5.6 Consulta conjunta de 2024 y 2026

Se crea la vista `viajes` (con el ajuste anterior) y se ejecuta una consulta que agrupa por año y tipo de taxi.

| | |
|---|---|
| **Objetivo** | Comprobar que una sola consulta devuelve datos de ambos años. |
| **Fuente** | Vista `viajes` (`data/raw/yellow/*/*.parquet`, `data/raw/green/*/*.parquet`) |
| **Consulta** | `sql/05_incorporacion/05_viajes_por_anio.sql` |

In [8]:
consulta("sql/04_eda/00_vista_viajes.sql")

-- Vista `viajes`: taxis amarillos y verdes unificados y filtrados.
-- Fuente: data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet (todos los anios descargados).
-- Lee los Parquet en cada consulta; no materializa datos.
--
-- Columnas: nombres comunes en snake_case, `tipo` ('yellow' | 'green'), `periodo` (mes del
-- archivo, 'AAAA-MM') y `duracion_min`. `airport_fee` solo existe en yellow y `trip_type` en green.
-- `cbd_congestion_fee` se reporta como 0 en los archivos que no tienen la columna (anteriores a 2025).
--
-- Filtros aplicados (ver notebooks/03_exploracion.ipynb):
--   - inicio del viaje dentro del mes del archivo;
--   - fin posterior al inicio y duracion <= 24 h;
--   - distancia > 0 y <= 100 millas;
--   - total_amount >= 0.
CREATE OR REPLACE VIEW viajes AS
WITH unificado AS (
    SELECT
        'yellow'                AS tipo,
        filename,
        VendorID                AS vendor_id,
        tpep_pickup_datetime    AS pickup_datetime,
        tpep_dropoff_dat

In [9]:
consulta(DIR_SQL / "05_viajes_por_anio.sql")

-- Ejercicio 5.6 - Consulta conjunta 2024 + 2026 sobre la vista `viajes`.
-- Fuente: vista `viajes` (data/raw/yellow/*/*.parquet y data/raw/green/*/*.parquet).
-- Salida: por anio y tipo, viajes, meses distintos, rango de fechas y metricas basicas.
SELECT
    year(pickup_datetime)                     AS anio,
    tipo,
    count(*)                                  AS viajes,
    count(DISTINCT periodo)                   AS meses,
    min(pickup_datetime)                      AS primer_viaje,
    max(pickup_datetime)                      AS ultimo_viaje,
    round(count(*) / count(DISTINCT pickup_datetime::DATE), 0) AS viajes_por_dia,
    round(avg(trip_distance), 2)              AS distancia_promedio,
    round(avg(total_amount), 2)               AS total_promedio
FROM viajes
GROUP BY anio, tipo
ORDER BY tipo, anio;



,anio,tipo,viajes,meses,primer_viaje,ultimo_viaje,viajes_por_dia,distancia_promedio,total_promedio
0,2024,green,623850,12,2024-01-01 00:03:57,2024-12-31 23:56:49,1705.0,2.95,24.21
1,2026,green,324160,8,2026-01-01 00:03:27,2026-08-31 23:58:28,1334.0,3.36,25.53
2,2024,yellow,39830199,12,2024-01-01 00:00:00,2024-12-31 23:59:58,108826.0,3.42,28.56
3,2026,yellow,28242805,8,2026-01-01 00:00:00,2026-08-31 23:59:59,116226.0,3.52,30.24


**Resultado:**

- La misma consulta devuelve **ambos años**: 2024 con 12 meses (1 de enero a 31 de diciembre) y 2026 con 8 meses (1 de enero a 31 de agosto), para los dos tipos de taxi.
- La vista conserva 39,830,199 viajes amarillos y 623,850 verdes de 2024.
- No hubo que cambiar la lectura de los archivos: los patrones `data/raw/yellow/*/*.parquet` y `data/raw/green/*/*.parquet` incluyeron automáticamente la nueva carpeta `2024/`.

## 5.7 ¿Las consultas anteriores necesitan modificarse?

Se ejecutan **sin cambios** todas las consultas de los Ejercicios 3 y 4 sobre el conjunto ampliado, y se registra si terminan correctamente, cuántas filas devuelven y cuánto tardan.

In [10]:
resultados = []
for ruta in sorted(Path("sql/03_exploracion").glob("*.sql")) + sorted(Path("sql/04_eda").glob("0[1-9]*.sql")) + sorted(Path("sql/04_eda").glob("1*.sql")):
    inicio = time.perf_counter()
    try:
        df = con.sql(ruta.read_text()).df()
        resultados.append({
            "consulta": str(ruta),
            "estado": "OK",
            "filas": len(df),
            "segundos": round(time.perf_counter() - inicio, 1),
        })
    except Exception as error:
        resultados.append({"consulta": str(ruta), "estado": f"ERROR: {error}", "filas": None,
                           "segundos": round(time.perf_counter() - inicio, 1)})
pd.DataFrame(resultados)

,consulta,estado,filas,segundos
0,sql/03_exploracion/01_cantidad_archivos.sql,OK,4,0.0
1,sql/03_exploracion/02_cantidad_registros.sql,OK,40,0.0
2,sql/03_exploracion/03_columnas.sql,OK,25,0.0
3,sql/03_exploracion/04_tipos_green.sql,OK,22,0.0
4,sql/03_exploracion/04_tipos_yellow.sql,OK,21,0.0
5,sql/03_exploracion/05_muestra_green.sql,OK,5,0.0
6,sql/03_exploracion/05_muestra_yellow.sql,OK,5,0.3
7,sql/03_exploracion/06_calidad.sql,OK,2,13.1
8,sql/03_exploracion/06_resumen_green.sql,OK,22,2.4
9,sql/03_exploracion/06_resumen_yellow.sql,OK,21,173.7


**Resultado:**

- Las **21 consultas** se ejecutaron correctamente (`OK`) sin modificar su código.
- Las consultas que agrupan por archivo o por mes ahora devuelven más filas, porque incluyen los períodos de 2024. Por ejemplo, `02_cantidad_registros.sql` devuelve 40 archivos y `02_viajes_por_mes.sql`, 40 combinaciones de mes y tipo.
- La consulta más lenta es `06_resumen_yellow.sql` (`SUMMARIZE` sobre todas las columnas de ~71 millones de registros amarillos), porque debe leer todo el conjunto de datos.

### Ejemplos de consultas reutilizadas

A continuación se muestran tres consultas de los ejercicios anteriores, ejecutadas sin cambios, para confirmar que sus resultados ya incluyen 2024.

In [11]:
consulta("sql/03_exploracion/01_cantidad_archivos.sql")

-- Ejercicio 3.1 - Cantidad de archivos Parquet disponibles.
-- Fuente: data/raw/*/*/*.parquet (todos los tipos de taxi y anios).
-- Salida: tipo de taxi, anio y cantidad de archivos.
SELECT
    split_part(file, '/', 3) AS tipo,
    split_part(file, '/', 4) AS anio,
    count(*)                 AS archivos
FROM glob('data/raw/*/*/*.parquet')
GROUP BY ALL
ORDER BY tipo, anio;



,tipo,anio,archivos
0,green,2024,12
1,green,2026,8
2,yellow,2024,12
3,yellow,2026,8


In [12]:
consulta("sql/04_eda/01_registros_filtrados.sql")

-- P0 - Registros conservados por la vista `viajes` frente al total de los archivos.
-- Fuente: data/raw/*/*/*.parquet (metadatos) y vista `viajes`.
WITH originales AS (
    SELECT split_part(file_name, '/', 3) AS tipo, sum(num_rows) AS registros_originales
    FROM parquet_file_metadata('data/raw/*/*/*.parquet')
    GROUP BY tipo
),
conservados AS (
    SELECT tipo, count(*) AS registros_conservados
    FROM viajes
    GROUP BY tipo
)
SELECT
    tipo,
    registros_originales,
    registros_conservados,
    registros_originales - registros_conservados                          AS registros_excluidos,
    round(100.0 * registros_conservados / registros_originales, 2)        AS pct_conservado
FROM originales
JOIN conservados USING (tipo)
ORDER BY tipo;



,tipo,registros_originales,registros_conservados,registros_excluidos,pct_conservado
0,green,997332.0,948010,49322.0,95.05
1,yellow,70873075.0,68073004,2800071.0,96.05


In [13]:
consulta("sql/04_eda/02_viajes_por_mes.sql").pivot(index="periodo", columns="tipo", values="viajes_por_dia")

-- P1 - Viajes por mes y tipo de taxi, con promedio diario.
-- Fuente: vista `viajes`.
SELECT
    periodo,
    tipo,
    count(*)                                                 AS viajes,
    round(count(*) / count(DISTINCT pickup_datetime::DATE), 0) AS viajes_por_dia
FROM viajes
GROUP BY periodo, tipo
ORDER BY tipo, periodo;



tipo,green,yellow
periodo,,
2024-01,1726.0,92644.0
2024-02,1744.0,100218.0
2024-03,1751.0,111439.0
2024-04,1771.0,114283.0
2024-05,1862.0,117070.0
2024-06,1729.0,114714.0
2024-07,1571.0,96205.0
2024-08,1578.0,92667.0
2024-09,1717.0,116668.0


**Resultado:**

- `01_cantidad_archivos.sql` (Ejercicio 3) ya muestra los archivos de 2024.
- `01_registros_filtrados.sql` (Ejercicio 4) cuenta los registros de ambos años: 70,873,075 amarillos y 997,332 verdes en los archivos, de los cuales la vista conserva 68,073,004 y 948,010.
- `02_viajes_por_mes.sql` (Ejercicio 4) devuelve los 20 meses de cada tipo de taxi.

**Conclusión de 5.7:** ninguna consulta necesitó cambios de código. El único ajuste fue en la vista (`coalesce` de `cbd_congestion_fee`), por un cambio en el contenido de los archivos y no en la estructura del sistema. Hay que tener en cuenta que las consultas del Ejercicio 4 que no agrupan por período ahora **combinan ambos años en un solo resultado**; para comparar años se debe agrupar por `year(pickup_datetime)` o por `periodo`.

## Resumen

- 2024 se obtuvo desde la fuente original con el mismo script, sin modificar su código: 24 archivos nuevos, 0 descargas repetidas.
- DuckDB consulta 2024 y 2026 de forma conjunta mediante patrones de archivos (`*/*.parquet`) y `union_by_name`.
- Las 21 consultas anteriores funcionan sin cambios. Solo se ajustó la vista para tratar la ausencia de `cbd_congestion_fee` en 2024.